# Sponsorship ROI calculator: messy deal export to three validated models

This notebook creates the full 5,000-row synthetic dataset behind the résumé claim, introduces spreadsheet-style defects, restores the exact clean contract, sends the cleaned data through the repository's real feature-engineering and training code, checks held-out metrics, computes SHAP importance, and runs all repository tests.

In [ ]:
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").is_file(), "Run this notebook from the repository root."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_DIR = ROOT / "notebooks" / "data"
ARTIFACT_DIR = ROOT / "notebooks" / "artifacts"
DATA_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Repository: {ROOT.name}")
print(f"Python: {sys.version.split()[0]}")

In [ ]:
import pandas as pd

from data.generate_data import generate_dataset

canonical = generate_dataset(n_records=5_000, seed=42)
messy = canonical.copy().astype(object)
messy["annual_spend"] = messy["annual_spend"].map(lambda value: f" ${float(value):,.0f} ")
for column in ["audience_reach", "social_following"]:
    messy[column] = messy[column].map(lambda value: f"{int(value):,}")
for column in ["sport", "market", "deal_type", "brand_category"]:
    messy[column] = messy[column].map(lambda value: f"  {str(value)}  ")
messy["analyst_note"] = "unstructured spreadsheet import"

invalid = messy.iloc[[0]].copy()
invalid["annual_spend"] = "unknown"
messy = pd.concat([messy, messy.iloc[:10], invalid], ignore_index=True)
messy = messy.sample(frac=1, random_state=42).reset_index(drop=True)
messy_path = DATA_DIR / "messy_sponsorship_deals.csv"
messy.to_csv(messy_path, index=False)
print(
    {
        "canonical_rows": len(canonical),
        "messy_rows": len(messy),
        "file": str(messy_path.relative_to(ROOT)),
    }
)
messy.head(3)

In [ ]:
cleaned = messy.drop(columns=["analyst_note"]).copy()
cleaned["annual_spend"] = pd.to_numeric(
    cleaned["annual_spend"].astype(str).str.replace(r"[$,\s]", "", regex=True),
    errors="coerce",
)
for column in ["audience_reach", "social_following"]:
    cleaned[column] = pd.to_numeric(
        cleaned[column].astype(str).str.replace(",", "", regex=False), errors="coerce"
    )
for column in ["sport", "market", "deal_type", "brand_category"]:
    cleaned[column] = cleaned[column].astype(str).str.strip()
cleaned = cleaned.dropna().drop_duplicates().reset_index(drop=True)
for column in [
    "deal_length_years",
    "audience_reach",
    "social_following",
    "on_air",
    "digital",
    "social",
    "experiential",
    "dooh",
]:
    cleaned[column] = cleaned[column].astype(int)
cleaned = cleaned[canonical.columns]

actual = cleaned.sort_values(list(canonical.columns)).reset_index(drop=True)
expected = canonical.sort_values(list(canonical.columns)).reset_index(drop=True)
pd.testing.assert_frame_equal(actual, expected, check_dtype=False)
clean_path = DATA_DIR / "cleaned_sponsorship_deals.csv"
cleaned.to_csv(clean_path, index=False)
print(f"Recovered all {len(cleaned):,} canonical deals.")

In [ ]:
import model.train as training

training.DATA_PATH = clean_path
training.MODEL_PATH = ARTIFACT_DIR / "trained_model.joblib"
training.METRICS_PATH = ARTIFACT_DIR / "metrics.json"
models, metrics = training.train_models()

assert set(models) == set(training.TARGETS)
assert metrics["roi_score"]["r2"] >= 0.77
print(metrics)

In [ ]:
import numpy as np
import shap

transformed = training.feature_engineering(cleaned)
feature_columns = [column for column in transformed.columns if column not in training.TARGETS]
sample = transformed[feature_columns].iloc[:100]
values = np.asarray(shap.TreeExplainer(models["roi_score"]).shap_values(sample))
importance = pd.Series(abs(values).mean(axis=0), index=feature_columns).sort_values(ascending=False)
assert values.shape == sample.shape
importance.head(10)

In [ ]:
completed = subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"],
    cwd=ROOT,
    text=True,
    capture_output=True,
    check=False,
)
print(completed.stdout)
print(completed.stderr)
assert completed.returncode == 0, "Repository tests failed"
print("VALIDATED: the repository test suite passes in this UV environment.")